# Hands-on RL Fine-tuning with verl on AMD Instinct™ MI300X

## Qwen3-4B LoRA GRPO Training on a Single GPU

In this workshop, we will run a small but complete RL fine-tuning workflow for a reasoning-style language model.

Modern LLMs are moving from chatbots to reasoning agents.  To support reasoning, tool use, and multi-step decision making, we use reinforcement learning after pretraining and supervised fine-tuning.

We will train **Qwen3-4B** with **LoRA + GRPO** using **verl** and **vLLM** on a single **AMD Instinct™ MI300X GPU**.

| Component | Role |
|---|---|
| **Qwen3-4B** | Base model |
| **LoRA** | Lightweight fine-tuning method |
| **GRPO** | Reward-based RL algorithm |
| **verl** | RL training framework |
| **vLLM** | Fast rollout generation engine |
| **AMD MI300X** | Single-GPU training platform |

## Why RL Workloads Fit AMD Instinct GPUs

Reinforcement learning workloads differ from ordinary supervised fine-tuning.

In RL post-training, the model repeatedly generates responses, scores those responses, and updates the policy. This means rollout generation can become a major part of the workload.

AMD Instinct MI300X is useful for this workshop because it provides:

- large HBM memory headroom
- high memory bandwidth
- strong compute capability
- ROCm software support
- enough room for model weights, LoRA adapters, rollout generation, and training overhead

This makes MI300X a strong fit for a single-GPU hands-on RL training demo.


## How verl and vLLM Work Together

In this training job, **verl controls the RL loop**, while **vLLM handles rollout generation**.

 | Component | Responsibility |
  |---|---|
  | **verl** | Orchestrates the RL loop: rollout scheduling, advantage computation, PPO/GRPO loss, optimizer step, weight sync, logging, checkpointing |
  | **vLLM** | High-throughput rollout engine — generates a *group* of responses per prompt for GRPO |
  | **Actor (LoRA)** | The policy being trained. Only LoRA adapter weights are updated; the base model stays frozen |
  | **Reference model** | Frozen copy of the initial policy, used as the KL anchor to keep the actor from drifting |
  | **GRPO** | Computes a group-relative advantage per response (per-prompt baseline), then applies a PPO-style clipped objective |
  | **Reward function** | Scores each generated response (rule-based, model-based, or a mix) |

### Runtime Flow

```text
 Prompt batch
    ↓
  vLLM rollout engine generates G responses per prompt (group sampling)
    ↓
  Reward function scores each response
    ↓
  GRPO computes group-relative advantages (per-prompt baseline)
    ↓
  Training engine recomputes log-probs; PPO-clipped loss + KL-to-reference
    ↓
  Optimizer updates LoRA adapters on the actor
    ↓
  verl syncs updated LoRA weights into the vLLM engine
    ↓
  Next training step

## Workshop Runtime Stack

| Layer | Setting |
|---|---|
| Model | `Qwen/Qwen3-4B` |
| Dataset | GSM8K |
| RL algorithm | `algorithm.adv_estimator=grpo` |
| Fine-tuning | LoRA — `actor_rollout_ref.model.lora_rank=32`, `actor_rollout_ref.model.lora_alpha=64` |
| Rollout backend | `actor_rollout_ref.rollout.name=vllm` |
| Number of samples | `actor_rollout_ref.rollout.n=8` |
| Tensor parallel size | `actor_rollout_ref.rollout.tensor_model_parallel_size=1` |
| Framework version | verl v0.7.1 |
| Rollout runtime | vLLM 0.18.0 with ROCm 7.2.1 |
  | Hardware | AMD Instinct MI300X (gfx942), single GPU |


 ## Step 1: Check AMD GPU

  Before launching verl, confirm the AMD GPUs are visible inside the container.

  Run `amd-smi list` to enumerate the devices. You should see one or more entries with:
  - **GPU-Name** containing `MI300X` (architecture `gfx942`)
  - ROCm version `7.2.1` reported in the header
  - **Mem-Usage** mostly free (e.g. `~3 GB / 196592 MB` per GPU when idle)

In [ ]:
!amd-smi

## Step 2: Verify verl Installation

  This workshop runs inside the prebuilt `rocm/verl` container, which already ships verl as an editable install at `/workspace/verl`.

  The cell below confirms the install and prints the version. The workshop targets **verl 0.7.1**.

  The training entrypoint used later is:

  ```bash
  python3 -m verl.trainer.main_ppo
  ```

  (GRPO and PPO share this entrypoint; the algorithm is selected via `algorithm.adv_estimator=grpo`.)

In [ ]:
import importlib, sys

try:
  import verl
  print("verl version :", verl.__version__)
  print("verl module  :", verl.__file__)
except ImportError:
  sys.exit(
      "verl is not installed. This notebook expects the rocm/verl image "
  )

 ## Step 3: Check vLLM and ROCm Runtime

  verl uses vLLM as the rollout engine on top of PyTorch + ROCm. The cell below confirms that:

  - **vLLM** is importable
  - **PyTorch** is the ROCm build (HIP runtime present)
  - At least one **AMD GPU** is visible to PyTorch

  If any of these fail, the training launch in Step 8 will not work.

  The relevant rollout config used later is:

  ```bash
  actor_rollout_ref.rollout.name=vllm
  actor_rollout_ref.rollout.tensor_model_parallel_size=1
  actor_rollout_ref.rollout.n=8
  ```

In [ ]:
import sys

import vllm
import torch

hip_version = getattr(torch.version, "hip", None)
device_count = torch.cuda.device_count() if torch.cuda.is_available() else 0

print("vLLM version    :", vllm.__version__)
print("PyTorch version :", torch.__version__)
print("HIP runtime     :", hip_version or "NOT a ROCm build")
print("GPUs visible    :", device_count)

assert hip_version, "PyTorch is not built with ROCm/HIP — install the ROCm wheel."
assert device_count > 0, "No AMD GPUs visible — check --device=/dev/dri --device=/dev/kfd."

# torch.cuda.get_device_name() returns '' for MI300X on some ROCm builds; use amd-smi for the marketing name.
print("OK: vLLM + ROCm runtime ready.")

  ## Step 4: Prepare the GSM8K Dataset

  We use **GSM8K** (grade-school math word problems) because it gives us a *deterministic* reward: each example ends with a `#### <number>` marker, so the GRPO reward function just extracts the model's final number and
  string-compares it to the gold answer. No reward model needed — perfect for a workshop.

  verl ships a preprocessing script at `examples/data_preprocess/gsm8k.py` that downloads GSM8K from Hugging Face and writes `train.parquet` / `test.parquet` for the trainer to consume.

  The output paths below (`/workspace/data/gsm8k/`) are what Step 7 will reference via `data.train_files=` and `data.val_files=`.

In [ ]:
%%bash
set -e

VERL_DIR=/workspace/verl
DATA_DIR=/workspace/data/gsm8k

if [ -f "${DATA_DIR}/train.parquet" ] && [ -f "${DATA_DIR}/test.parquet" ]; then
echo "GSM8K already prepared at ${DATA_DIR}, skipping."
else
mkdir -p "${DATA_DIR}"
python3 "${VERL_DIR}/examples/data_preprocess/gsm8k.py" --local_dir "${DATA_DIR}"
fi

echo "Files in ${DATA_DIR}:"
ls -lh "${DATA_DIR}"

## Step 5: Model and Dataset Paths

We point the trainer at two things:

- **Dataset** — the GSM8K parquets produced by Step 4, on the container's local filesystem.
- **Model** — Qwen3-4B base weights, pulled from Hugging Face into a host-mounted cache (`/root/.cache/huggingface` → `/data/huggingface` on the host) so they survive container restarts.


In [ ]:
from pathlib import Path

MODEL_PATH  = "Qwen/Qwen3-4B"
VERL_DIR    = Path("/workspace/verl")
DATA_DIR    = Path("/workspace/data/gsm8k")
train_files = str(DATA_DIR / "train.parquet")
test_files  = str(DATA_DIR / "test.parquet")

print("MODEL_PATH  :", MODEL_PATH)
print("VERL_DIR    :", VERL_DIR, "(exists:", VERL_DIR.exists(), ")")
print("train_files :", train_files, "(exists:", Path(train_files).exists(), ")")
print("test_files  :", test_files,  "(exists:", Path(test_files).exists(),  ")")

assert VERL_DIR.exists(), f"verl not found at {VERL_DIR} — re-check Step 2."

##### dataset: download if missing, then peek:

In [ ]:
import subprocess, sys
import pandas as pd

if not (Path(train_files).exists() and Path(test_files).exists()):
  print("GSM8K parquets not found — running the verl preprocessing script.")
  DATA_DIR.mkdir(parents=True, exist_ok=True)
  # Pulls openai/gsm8k from Hugging Face Datasets and writes train/test parquets.
  subprocess.run(
      [sys.executable, str(VERL_DIR / "examples/data_preprocess/gsm8k.py"),
       "--local_dir", str(DATA_DIR)],
      check=True,
  )
else:
  print(f"GSM8K parquets already present at {DATA_DIR}, skipping download.")

train_df = pd.read_parquet(train_files)
test_df  = pd.read_parquet(test_files)
print(f"\ntrain rows : {len(train_df):>6}   columns: {list(train_df.columns)}")
print(f"test  rows : {len(test_df):>6}   columns: {list(test_df.columns)}")

# Show one example so the reader sees what the trainer will consume.
example = train_df.iloc[0]
print("\n--- sample row ---")
for col in train_df.columns:
  val = str(example[col])
  print(f"{col}: {val[:300]}{'...' if len(val) > 300 else ''}")

##### model: download via from_pretrained if not in HF cache:

In [ ]:
import os
from pathlib import Path

# HF stores snapshots under hub/models--<org>--<name>/.
hf_cache  = Path(os.environ.get("HF_HOME", str(Path.home() / ".cache/huggingface"))) / "hub"
model_dir = hf_cache / f"models--{MODEL_PATH.replace('/', '--')}"

if not model_dir.exists() or not any(model_dir.rglob("*.safetensors")):
  print(f"{MODEL_PATH} not found in HF cache — downloading via transformers.")
  print("First run pulls ~8 GB; subsequent runs hit the cache.\n")

  import transformers
  # from_pretrained downloads config, tokenizer, and weight shards into the HF cache,
  # then materializes the model object. We keep it on CPU and discard immediately —
  # the goal here is only to populate the cache for the trainer in Step 8.
  tokenizer = transformers.AutoTokenizer.from_pretrained(MODEL_PATH)
  model     = transformers.AutoModelForCausalLM.from_pretrained(
      MODEL_PATH,
      torch_dtype="auto",
      device_map="cpu",
  )
  del model, tokenizer
else:
  print(f"{MODEL_PATH} already cached at {model_dir}, skipping download.")

# Show what's in the snapshot so the reader sees what an LLM repo actually contains.
snapshot_root = next((model_dir / "snapshots").iterdir(), None)
if snapshot_root is not None:
  print(f"\nFiles in snapshot ({snapshot_root.name}):")
  for p in sorted(snapshot_root.iterdir()):
      size_mb = p.stat().st_size / (1024 * 1024) if p.is_file() else 0
      print(f"  {p.name:<40} {size_mb:>8.1f} MB" if p.is_file() else f"  {p.name}/")

## Step 6: Understand the Key Training Configuration

Step 7 launches `verl.trainer.main_ppo` with a long list of Hydra overrides. Here are the ones that actually shape this run; everything else is plumbing.

### Algorithm

| Config | Meaning |
|---|---|
| `algorithm.adv_estimator=grpo` | Compute group-relative advantages instead of GAE — this is what makes the run GRPO instead of vanilla PPO |
| `actor_rollout_ref.actor.use_kl_loss=true` | Add a KL penalty against the frozen reference model (anchors the policy, prevents collapse) |
| `actor_rollout_ref.actor.kl_loss_coef=0.001` | Strength of that KL penalty |

### Model and LoRA

| Config | Meaning |
|---|---|
| `actor_rollout_ref.model.path=Qwen/Qwen3-4B` | Base model — pulled from the HF cache populated in Step 5 |
| `actor_rollout_ref.model.lora_rank=32` | Rank of the LoRA matrices A (d × 32) and B (32 × d). Higher = more capacity, more trainable params |
| `actor_rollout_ref.model.lora_alpha=64` | LoRA scaling. Effective scale = `alpha / rank = 2.0` |

### Rollout (vLLM)

| Config | Meaning |
|---|---|
| `actor_rollout_ref.rollout.name=vllm` | Use vLLM as the rollout engine |
| `actor_rollout_ref.rollout.n=8` | Sample 8 responses per prompt — this is the GRPO group size |
| `actor_rollout_ref.rollout.tensor_model_parallel_size=1` | vLLM does not shard the model across GPUs |
| `actor_rollout_ref.rollout.gpu_memory_utilization=0.5` | Fraction of GPU memory vLLM may use for its KV cache. The remainder is reserved for the trainer's actor + optimizer state + activations — essential when vLLM and
training share one GPU |

### Training engine

| Config | Meaning |
|---|---|
| `actor_rollout_ref.actor.strategy=fsdp2` | Run the actor under Fully Sharded Data Parallel v2. On a single GPU there's nothing to shard across, but FSDP2 still manages activation and gradient memory efficiently |
| `actor_rollout_ref.ref.strategy=fsdp2` | Same for the frozen reference policy used by the KL term |

### Data and schedule

| Config | Meaning |
|---|---|
| `data.train_files=/workspace/data/gsm8k/train.parquet` | Training prompts — produced in Step 4 |
| `data.val_files=/workspace/data/gsm8k/test.parquet` | Eval prompts |
| `data.train_batch_size=...` | Number of prompts sampled per RL step (each is expanded into `n=8` rollouts) |
| `actor_rollout_ref.actor.optim.lr=...` | Learning rate for the LoRA adapter optimizer |
| `trainer.total_epochs=...` (or `trainer.total_training_steps=...`) | How long training runs |

## Step 7: Launch Qwen3-4B LoRA GRPO Training

### What spins up at launch

When the cell runs, verl will:

1. Initialize Ray (single-node) and spawn worker processes
2. Load **Qwen3-4B** as the actor under FSDP2 and attach LoRA adapters (rank 32)
3. Load a frozen **reference model** copy under FSDP2 for the KL anchor
4. Boot a **vLLM** rollout engine inside the actor worker
5. Run a baseline eval pass (`val_before_train=True`)
6. Enter the GRPO training loop: rollout → reward → advantages → PPO-clipped + KL update → sync LoRA weights into vLLM → next step
7. Stream metrics to the console and to TensorBoard

### To make the run shorter (workshop time control)

Reduce any of: `trainer.total_epochs`, `trainer.test_freq`, `data.train_batch_size`, `actor_rollout_ref.rollout.n`, `data.max_response_length`.

> Note: `data.shuffle=False` makes the prompt order deterministic for reproducibility in this workshop. For real training runs, set it to `True`.

In [ ]:
%%bash
python3 - <<'PY'
from pathlib import Path
p = Path("/workspace/verl/verl/utils/tracking.py")
src = p.read_text()
old = 'self.fp.write(json.dumps(data) + "\\n")'
new = (
  "import torch as _torch\n"
  "        data = {k: (v.item() if isinstance(v, _torch.Tensor) and v.numel()==1 "
  "else v.tolist() if isinstance(v, _torch.Tensor) else v) for k, v in data.items()}\n"
  "        " + old
)
if old not in src:
  print("Pattern not found — file may have a different format. Open and patch manually.")
elif new.split('\n')[0] in src:
  print("Already patched.")
else:
  p.write_text(src.replace(old, new))
  print("Patched. Restart training.")
PY

In [ ]:
%%bash
set -e

export HIP_VISIBLE_DEVICES=0
export GPUS_PER_NODE=1

cd /workspace/verl

TIMESTAMP=$(date +%Y%m%d.%H%M%S)
project_name=grpo_qwen_llm
experiment_name=qwen3_4b_grpo_lora_rocm_single_gpu_${TIMESTAMP}
train_dir=/workspace/outputs/${project_name}/${experiment_name}
mkdir -p "${train_dir}"
export TENSORBOARD_DIR="${train_dir}/tensorboard_log"
export VERL_FILE_LOGGER_PATH="${train_dir}/metrics.jsonl"

# Sentinel so the monitor cells can find this run.
echo "${train_dir}" > /tmp/verl_last_run

MODEL_PATH="Qwen/Qwen3-4B"
train_files="/workspace/data/gsm8k/train.parquet"
test_files="/workspace/data/gsm8k/test.parquet"

GPU_MEMORY_UTILIZATION=0.3
max_token_len_per_gpu=24576
TP_VALUE=1
TRAIN_BATCH_SIZE=16
MINI_BATCH_SIZE=16

nohup python3 -m verl.trainer.main_ppo \
algorithm.adv_estimator=grpo \
trainer.val_before_train=True \
data.train_files="${train_files}" \
data.val_files="${test_files}" \
data.train_batch_size=${TRAIN_BATCH_SIZE} \
data.max_prompt_length=1024 \
data.max_response_length=1024 \
data.filter_overlong_prompts=True \
data.truncation='error' \
data.shuffle=False \
actor_rollout_ref.model.path="${MODEL_PATH}" \
actor_rollout_ref.model.use_remove_padding=True \
actor_rollout_ref.model.enable_gradient_checkpointing=True \
actor_rollout_ref.model.lora_rank=32 \
actor_rollout_ref.model.lora_alpha=64 \
actor_rollout_ref.actor.optim.lr=1.0e-05 \
actor_rollout_ref.actor.use_dynamic_bsz=True \
actor_rollout_ref.actor.ppo_mini_batch_size=${MINI_BATCH_SIZE} \
actor_rollout_ref.actor.ppo_max_token_len_per_gpu=${max_token_len_per_gpu} \
actor_rollout_ref.actor.use_kl_loss=True \
actor_rollout_ref.actor.kl_loss_coef=0.001 \
actor_rollout_ref.actor.kl_loss_type=low_var_kl \
actor_rollout_ref.actor.entropy_coeff=0 \
actor_rollout_ref.actor.strategy=fsdp2 \
actor_rollout_ref.actor.fsdp_config.model_dtype=bf16 \
actor_rollout_ref.actor.fsdp_config.param_offload=False \
actor_rollout_ref.actor.fsdp_config.optimizer_offload=False \
actor_rollout_ref.rollout.log_prob_use_dynamic_bsz=True \
actor_rollout_ref.rollout.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
actor_rollout_ref.rollout.tensor_model_parallel_size=${TP_VALUE} \
actor_rollout_ref.rollout.name=vllm \
actor_rollout_ref.rollout.gpu_memory_utilization=${GPU_MEMORY_UTILIZATION} \
actor_rollout_ref.rollout.n=8 \
actor_rollout_ref.rollout.load_format=safetensors \
actor_rollout_ref.rollout.layered_summon=True \
actor_rollout_ref.ref.log_prob_use_dynamic_bsz=True \
actor_rollout_ref.ref.log_prob_max_token_len_per_gpu=${max_token_len_per_gpu} \
actor_rollout_ref.ref.fsdp_config.param_offload=True \
actor_rollout_ref.ref.strategy=fsdp2 \
actor_rollout_ref.ref.fsdp_config.model_dtype=bf16 \
algorithm.use_kl_in_reward=False \
trainer.use_legacy_worker_impl=disable \
trainer.critic_warmup=0 \
trainer.logger='["console","tensorboard"]' \
trainer.project_name="${project_name}" \
trainer.experiment_name="${experiment_name}" \
trainer.default_local_dir="${train_dir}" \
trainer.n_gpus_per_node=${GPUS_PER_NODE} \
trainer.nnodes=1 \
trainer.save_freq=20 \
trainer.test_freq=10 \
trainer.total_training_steps=40 \
trainer.total_epochs=1 \
> "${train_dir}/train_log.txt" 2>&1 &

echo $! > "${train_dir}/train.pid"
disown

echo "Training started"
echo "  PID      : $(cat ${train_dir}/train.pid)"
echo "  Log file : ${train_dir}/train_log.txt"
echo "  Run dir  : ${train_dir}"

### How long will this run, and where do checkpoints land?

**Step budget.** The cell sets `trainer.total_training_steps=40` — training stops after 40 GRPO updates, regardless of how much of the dataset has been seen. Each step does: vLLM rollout (16 prompts × 8 samples = 128
generations) → reward → GRPO update → LoRA weight sync back to vLLM. On a single MI300X this is roughly **30–60 seconds per step**, so the workshop run takes **~20–40 minutes**.

**Tuning the run length.** Edit `trainer.total_training_steps` in the launch cell:

| Goal | `total_training_steps` | Wall-clock (approx) |
|---|---|---|
| Workshop demo — see reward curve move | **40** (current) | 20–40 min |
| Short real run — clear val score gain | 150 | 1.5–2.5 hr |
| Full epoch on GSM8K (~7473 / 16 prompts) | 467 (or set `total_epochs=1` and remove the steps cap) | 3–7 hr |

**Checkpoints.** Written to `${train_dir}/global_step_<N>/` — each contains the LoRA adapter, optimizer state, and FSDP shards (a few GB per checkpoint). The cell uses `trainer.save_freq=20`, which for a 40-step run
writes:

- `global_step_20/` — mid-run snapshot
- `global_step_40/` — final

Tune `save_freq` together with `total_training_steps`. Aim for **3–5 checkpoints across the full run**:

| `total_training_steps` | Suggested `save_freq` | Checkpoints written |
|---|---|---|
| 40 | 20 (current) | 2 |
| 150 | 50 | 3 |
| 467 | 100 | 4–5 |

Two practical notes:
- `save_freq=-1` disables periodic saving but verl still writes the final checkpoint — use this if you only care about the trained adapter at the end.
- Aligning `save_freq` with `trainer.test_freq` (currently 10) means each saved checkpoint sits next to a recorded validation score, making it easy to pick the best one later.

### Watch training progress live

The cell below tails `train_log.txt` from the background training process and refreshes every 3 seconds, so you can follow rollout speed, reward, KL, and val scores as they're produced.

**This is a viewer, not the trainer.** Training was launched in the previous cell with `nohup` — it runs in the background and is independent of this cell. Watching or not watching has no effect on the run.

**To stop watching** (without affecting training):
- **Jupyter / JupyterLab**: click the ■ Stop button in the toolbar, or press `I, I` (interrupt kernel)
- **VS Code notebooks**: click the ■ Interrupt button next to the cell
- **Any environment**: menu `Kernel → Interrupt`

You'll see `(stopped watching — training continues in background)`. Re-run the cell anytime to resume tailing — the log file is the same.

**The cell also exits on its own** once the training process finishes (it checks `/proc/<pid>` each tick), so for the 40-step run you can leave it running and it will stop ~30 minutes later when training completes.

**To actually stop training**, use the explicit stop cell further down — don't rely on interrupting this viewer.

In [ ]:
import time, subprocess
from pathlib import Path
from IPython.display import clear_output

train_dir = Path(Path("/tmp/verl_last_run").read_text().strip())
log_file  = train_dir / "train_log.txt"
pid_file  = train_dir / "train.pid"
pid       = int(pid_file.read_text().strip())

def alive(pid):
  return Path(f"/proc/{pid}").exists()

try:
  while True:
      out = subprocess.run(
          ["tail", "-n", "60", str(log_file)],
          capture_output=True, text=True,
      ).stdout
      clear_output(wait=True)
      status = "RUNNING" if alive(pid) else "EXITED"
      print(f"[{status}] PID={pid}  log={log_file}\n" + "-" * 80)
      print(out)
      if not alive(pid):
          break
      time.sleep(3)
except KeyboardInterrupt:
  print("\n(stopped watching — training continues in background)")

### Stop the current training run

Use the cell below to stop training **before it finishes** — e.g. to edit the launch cell and restart with new settings, or to free the GPU for inference.

**You don't need to stop** for editing the plotter or interrupting the log-tail cell — those are just viewers.

**What the cell does:** sends `TERM` to the launcher, falls back to `KILL` after 5s, then sweeps the Ray workers and vLLM engine (they run as siblings, not children, so the launcher PID alone doesn't catch them).

**To restart with new settings:** edit the launch cell and re-run it. A fresh `TIMESTAMP` produces a new `${train_dir}`, and `/tmp/verl_last_run` is overwritten — all viewer cells pick up the new run automatically.

> Stopping mid-run discards everything since the last checkpoint. With `save_freq=20`, stopping at step 35 leaves you at `global_step_20/`.

In [ ]:
%%bash
set -e

train_dir=$(cat /tmp/verl_last_run)
pid=$(cat "${train_dir}/train.pid")

echo "Stopping training: pid=${pid}, run=${train_dir}"

# 1. TERM the launcher (graceful — gives Ray a chance to clean up).
kill -TERM "${pid}" 2>/dev/null && echo "  sent TERM to ${pid}" || echo "  ${pid} already gone"

# 2. Give it a few seconds to exit, then KILL if still alive.
for i in 1 2 3 4 5; do
if [ ! -d "/proc/${pid}" ]; then break; fi
sleep 1
done
if [ -d "/proc/${pid}" ]; then
kill -KILL "${pid}" 2>/dev/null && echo "  sent KILL to ${pid}"
fi

# 3. Sweep up Ray workers and vLLM engine processes verl spawned.
#    These are siblings, not children of the launcher, so kill -- -PID misses them.
pkill -KILL -f "verl.trainer.main_ppo" 2>/dev/null || true
pkill -KILL -f "ray::"                  2>/dev/null || true
pkill -KILL -f "vllm"                   2>/dev/null || true

# 4. Verify nothing is still holding GPU memory.
echo ""
echo "--- remaining verl/ray/vllm processes (should be empty) ---"
pgrep -af "verl.trainer.main_ppo|ray::|vllm" || echo "(none)"
echo ""
echo "--- GPU 0 status ---"
amd-smi monitor -p -g 0 2>/dev/null | head -5 || rocm-smi --showuse 2>/dev/null | head

## Step 8: Run Inference with the Trained Checkpoint

Load the trained model and have it solve a held-out GSM8K-style problem — does GRPO actually change the model's behavior?

**What the cell does:**
1. Finds the latest `global_step_<N>/` checkpoint.
2. Merges FSDP shards + LoRA adapter into HF format via `verl.model_merger` (cached at `${checkpoint}/actor/merged_hf/`, ~30–60s first time only).
3. Loads with `transformers` and generates an answer with greedy decoding.

**Compare against the base model** by re-running with `MODEL_PATH = "Qwen/Qwen3-4B"` instead of `merged_dir`. After GRPO you should see cleaner reasoning, reliable `####` formatting, and more correct answers.

**Needs:** at least one saved checkpoint (first lands at step 20 with `save_freq=20`) and ~10 GB free on GPU 0.

In [ ]:
"""
  Load a saved checkpoint and run a sample GSM8K-style prompt.

  Verl writes each checkpoint as FSDP shards under:
    ${train_dir}/global_step_<N>/actor/

  For inference we first merge the shards (and the LoRA adapter, since this is a
  LoRA run) into a standard Hugging Face model directory, then load it with
  transformers. The merge step is one-shot and cached — re-running this cell uses
  the existing merged copy.
"""
import subprocess
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Locate the latest checkpoint from the most recent run.
train_dir = Path(Path("/tmp/verl_last_run").read_text().strip())
ckpts = sorted(train_dir.glob("global_step_*"),
             key=lambda p: int(p.name.split("_")[-1]))
assert ckpts, (
  f"No checkpoints in {train_dir}. Set trainer.save_freq > 0 and let "
  f"training run at least that many steps before retrying."
)
latest = ckpts[-1]
actor_dir = latest / "actor"
merged_dir = actor_dir / "merged_hf"
print(f"Latest checkpoint : {latest.name}   (step {latest.name.split('_')[-1]})")
print(f"FSDP shards dir   : {actor_dir}")
print(f"Target HF dir     : {merged_dir}")

# 2. Merge FSDP shards + LoRA adapter -> HF format (skip if already done).
if not (merged_dir / "config.json").exists():
  print("\nMerging FSDP shards + LoRA adapter into HF format...")
  subprocess.run([
      "python3", "-m", "verl.model_merger", "merge",
      "--backend", "fsdp",
      "--local_dir", str(actor_dir),
      "--target_dir", str(merged_dir),
  ], check=True)
  print(f"Merged model written to: {merged_dir}")
else:
  print(f"\nAlready merged at {merged_dir}, skipping merge step.")

# 3. Load the merged model and run a sample prompt.
print("\nLoading merged model on GPU 0...")
tok = AutoTokenizer.from_pretrained(str(merged_dir))
model = AutoModelForCausalLM.from_pretrained(
  str(merged_dir), torch_dtype=torch.bfloat16, device_map="cuda:0"
)
model.eval()

# A held-out style GSM8K problem (not from the training set).
question = (
  "A robot needs 7 batteries to run for one hour. "
  "If the robot ran for 5 hours, how many batteries did it use? "
  'Let\'s think step by step and output the final answer after "####".'
)

inputs = tok.apply_chat_template(
  [{"role": "user", "content": question}],
  add_generation_prompt=True,
  return_tensors="pt",
).to("cuda:0")

with torch.no_grad():
  out = model.generate(
      inputs,
      max_new_tokens=512,
      do_sample=False,        # greedy → deterministic, matches GRPO eval
      temperature=0.0,
      pad_token_id=tok.eos_token_id,
  )

response = tok.decode(out[0][inputs.shape[1]:], skip_special_tokens=True)
print("\n" + "=" * 70)
print("Question:", question)
print("=" * 70)
print(response)